# 在 Colab 使用 IndexTTS 2.0（固定版本）

本笔记本固定安装官方 **v2.0.0**，不会跟随 `main` 自动升级到 2.5。

1. 在「运行时 → 更改运行时类型」选择 **T4 GPU**，然后连接。
2. 从上到下运行全部单元格。首次需要下载依赖及模型，请留出时间和磁盘空间。
3. 最后一格出现 `https://…gradio.live` 后，打开链接，上传参考音频、输入文字并生成语音。

如果此前运行过 2.5，建议先保存音频，再用「运行时 → 断开连接并删除运行时」重新连接。
本笔记本使用独立目录 `/content/index-tts-2.0-pinned`，不会删除旧版目录。

- 官方代码：[v2.0.0](https://github.com/index-tts/index-tts/tree/830f6f8f94a51fea23ab1d639027a86200075a4e)
- 主模型：[IndexTeam/IndexTTS-2，2025-09-08 快照](https://huggingface.co/IndexTeam/IndexTTS-2/tree/258515cc44cee99d5b9694a67ee194ffd8a3e618)
- 基于 [xcrong 的 Colab 笔记本](https://github.com/xcrong/free-indextts-1.5-on-colab) 改写。

Colab 免费 GPU 的可用性和时长由 Google 决定。Gradio 分享链接可被持有链接的人访问；用完停止最后一格，及时下载生成的音频。

## 1. 检查 GPU、安装工具

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

TOOLS_READY = False
SOURCE_READY = False
ENV_READY = False
MODEL_READY = False
SOURCE_SHA = "830f6f8f94a51fea23ab1d639027a86200075a4e"
MODEL_REPO = "IndexTeam/IndexTTS-2"
MODEL_SHA = "258515cc44cee99d5b9694a67ee194ffd8a3e618"
ROOT = Path("/content/index-tts-2.0-pinned")
UV = [sys.executable, "-m", "uv"]

def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

if not Path("/content").is_dir() or not shutil.which("nvidia-smi"):
    raise RuntimeError("请在 Colab 的「运行时 → 更改运行时类型」选择 T4 GPU 后重新运行。")
run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "git-lfs", "ffmpeg"])
run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.8.22"])
run(UV + ["--version"])
TOOLS_READY = True

## 2. 下载并核对官方 v2.0.0 代码
仅获取指定提交，不提供“最新版本”选项。重复运行会复用已下载目录；发现其他版本会停止。

In [ ]:
if not TOOLS_READY:
    raise RuntimeError("请先完成第 1 步。")
SOURCE_READY = False
if not ROOT.exists():
    ROOT.mkdir(parents=True)
if not (ROOT / ".git").exists():
    if any(ROOT.iterdir()):
        raise RuntimeError(f"{ROOT} 已存在且不是本笔记本的代码目录。请使用全新的 Colab 运行时。")
    run(["git", "init", ROOT])
    run(["git", "-C", ROOT, "remote", "add", "origin", "https://github.com/index-tts/index-tts.git"])
head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
if head.returncode != 0:
    run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", SOURCE_SHA])
    run(["git", "-C", ROOT, "checkout", "--detach", SOURCE_SHA], env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
actual = run(["git", "-C", ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if actual != SOURCE_SHA:
    raise RuntimeError(f"版本不匹配：{actual}。请使用全新的 Colab 运行时。")
run(["git", "-C", ROOT, "diff", "--exit-code", "HEAD", "--", "pyproject.toml", "uv.lock", "webui.py", "indextts"])
run(["git", "-C", ROOT, "lfs", "install", "--local"])
run(["git", "-C", ROOT, "lfs", "pull"])
SOURCE_READY = True
print(f"官方 IndexTTS v2.0.0 代码已核对：{actual}")

## 3. 安装固定依赖，检查版本与 CUDA

使用独立的 **Python 3.11** 环境及官方 `uv.lock`，避免 Colab 自带 Python 升级造成旧依赖不兼容。
只安装 WebUI 所需附加依赖；默认不启用 DeepSpeed 和自定义 CUDA 编译。

In [ ]:
if not SOURCE_READY:
    raise RuntimeError("请先完成第 2 步。")
ENV_READY = False
run(UV + ["python", "install", "3.11"])
run(UV + ["sync", "--frozen", "--python", "3.11", "--extra", "webui"], cwd=ROOT)
PYTHON = ROOT / ".venv/bin/python"
check_env = """
import sys, tomllib
from importlib.metadata import version
import torch
with open('pyproject.toml', 'rb') as f:
    project = tomllib.load(f)['project']
assert project['version'] == '2.0.0', project['version']
assert version('indextts') == '2.0.0', version('indextts')
assert sys.version_info[:2] == (3, 11), sys.version
assert torch.cuda.is_available(), 'CUDA 不可用，请确认已选择 GPU 运行时。'
x = torch.ones(1, device='cuda')
assert (x + x).item() == 2
print('IndexTTS:', version('indextts'))
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
"""
run([PYTHON, "-c", check_env], cwd=ROOT)
ENV_READY = True

## 4. 下载固定的 IndexTTS-2 模型

主模型固定到 2.0 发布时的快照；下载中断后可重跑此格续传。
官方推理代码还会在首次启动时自动下载 W2V-BERT、MaskGCT、CAMPPlus 和 BigVGAN 辅助模型；这些辅助模型仍沿用官方下载逻辑。

In [ ]:
if not ENV_READY:
    raise RuntimeError("请先完成第 3 步。")
MODEL_READY = False
download_model = """
import sys
from pathlib import Path
from huggingface_hub import snapshot_download
from omegaconf import OmegaConf
repo_id, revision = sys.argv[1:3]
snapshot_download(repo_id=repo_id, revision=revision, local_dir='checkpoints')
root = Path('checkpoints')
cfg = OmegaConf.load(root / 'config.yaml')
assert str(cfg.version) == '2.0', f'模型版本不匹配：{cfg.version}'
required = [cfg.dataset.bpe_model, cfg.gpt_checkpoint, cfg.s2mel_checkpoint,
            cfg.w2v_stat, cfg.emo_matrix, cfg.spk_matrix]
for name in required:
    p = root / name
    assert p.is_file() and p.stat().st_size > 0, f'缺少模型文件：{p}'
emo = root / cfg.qwen_emo_path
assert (emo / 'config.json').is_file(), f'缺少情绪模型配置：{emo}'
assert any(emo.glob('*.safetensors')), f'缺少情绪模型权重：{emo}'
(root / '.indextts2-revision').write_text(revision, encoding='utf-8')
print('IndexTTS 模型版本:', cfg.version)
print('主模型快照:', revision)
"""
run([PYTHON, "-c", download_model, MODEL_REPO, MODEL_SHA], cwd=ROOT)
MODEL_READY = True

## 5. 启动 WebUI

等待模型加载完成后，打开输出中的 **gradio.live** 链接。此格会持续运行，这是正常现象。
默认开启 FP16 以节省显存。若显存不足，先用较短文本和较短的干净参考音频测试。
需要重启界面时，停止此格后再次运行。

In [ ]:
if not MODEL_READY:
    raise RuntimeError("请先完成第 4 步。")
actual = run(["git", "-C", ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if actual != SOURCE_SHA or (ROOT / "checkpoints/.indextts2-revision").read_text().strip() != MODEL_SHA:
    raise RuntimeError("代码或模型版本不匹配，请重新从第 1 步运行。")
source = (ROOT / "webui.py").read_text(encoding="utf-8")
original = "demo.launch(server_name=cmd_args.host, server_port=cmd_args.port)"
replacement = "demo.launch(server_name=cmd_args.host, server_port=cmd_args.port, share=True)"
if source.count(original) != 1:
    raise RuntimeError("WebUI 启动代码与固定版本不一致，已停止，避免错误启动。")
launcher = ROOT / "webui_colab_2_0.py"
launcher.write_text(source.replace(original, replacement), encoding="utf-8")
print("启动 IndexTTS 2.0（FP16）。等待出现 gradio.live 分享链接。", flush=True)
run([PYTHON, "-u", launcher, "--fp16", "--model_dir", ROOT / "checkpoints"],
    cwd=ROOT, env={**os.environ, "PYTHONUNBUFFERED": "1"})

## 常见问题

- **为什么旧笔记本会安装 2.5？** 旧版从官方最近提交中选择代码，默认最新提交；标题不决定实际版本。本版锁定完整提交 SHA。
- **出现安装或下载错误：** 修复报错后重跑对应格，再继续后面的步骤。代码在子进程失败时立即停止，不会把失败显示为安装成功。
- **没有 GPU / CUDA 检查失败：** 确认选择 T4 GPU；如果免费额度不足，需要等待 Colab 再次分配 GPU。
- **打开界面很慢：** 首次启动还要下载辅助模型，查看最后一格的下载进度。
- **分享链接无法打开：** 确认最后一格仍在运行；停止后重新运行生成新链接。Gradio 分享服务和 Colab 网络需可用。
- **保存结果：** 在 WebUI 下载音频；运行时被删除后，`/content` 内的临时文件会消失。

验证范围：已做笔记本结构、代码语法、固定版本来源与启动补丁检查；尚未在真实 Colab GPU 上完成模型加载及语音生成验证。